# 地震属性与 gain 关系：绝对 gain vs 相对 gain

本 notebook 只读取 `wavelet_amplitude_gain_curve_depth@20260428.ipynb` 的输出，按井内有效样点自适应切段，比较地震振幅属性与不同 gain 目标之间的关系。

对每个 segment 同时输出两类 gain 目标：

- `segment_ls`：在同一 segment 内重新做局部最小二乘得到的 gain。
- `segment_median`：同一 segment 内已有 `gain_smooth` 的中位数。

每类 gain 都比较 absolute 与 relative 两种形式：

- absolute: `log(gain)`
- relative: `log(gain / scale)`


In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
if not (repo_root / "src").exists():
    raise RuntimeError("Could not locate repo root containing 'src'.")

plt.rcParams["figure.dpi"] = 120
pd.set_option("display.max_columns", 120)


## 1) 配置

In [ ]:
data_root = repo_root / "data"

gain_curve_output_dir = data_root / "output_wavelet_amplitude_gain_curve_depth_20260428"
gain_curve_metrics_file = gain_curve_output_dir / "wavelet_amplitude_gain_curve_metrics.csv"

output_dir = data_root / "output_wavelet_amplitude_gain_abs_vs_relative_attribute_20260429"
figure_dir = output_dir / "figures"
segment_samples_file = output_dir / "wavelet_amplitude_gain_abs_vs_relative_segment_samples.csv"
relationship_metrics_file = output_dir / "wavelet_amplitude_gain_abs_vs_relative_relationship_metrics.csv"
best_relationships_file = output_dir / "wavelet_amplitude_gain_abs_vs_relative_best_relationships.csv"

# Adaptive segmentation parameters.
min_segment_valid_samples = 8
max_segment_count = 20
min_segments_per_well = 1

# Optional quality gate. Leave as None to include all successful gain curves.
min_source_corr = None

required_paths = [gain_curve_metrics_file]
for path in required_paths:
    if not path.exists():
        raise FileNotFoundError(path)

output_dir.mkdir(parents=True, exist_ok=True)
figure_dir.mkdir(parents=True, exist_ok=True)

gain_metrics_df = pd.read_csv(gain_curve_metrics_file)
ok_gain_df = gain_metrics_df.loc[gain_metrics_df["status"] == "ok"].copy()
if min_source_corr is not None:
    ok_gain_df = ok_gain_df.loc[ok_gain_df["source_corr"] >= float(min_source_corr)].copy()
if ok_gain_df.empty:
    raise ValueError("No gain curves available for attribute relationship QC.")

print("Gain metrics:", gain_curve_metrics_file)
print("Wells:", ok_gain_df["well_name"].tolist())
print("Output dir:", output_dir)


## 2) 工具函数

In [ ]:
def resolve_artifact_path(path_value: str | Path) -> Path:
    path = Path(path_value)
    return path if path.is_absolute() else repo_root / path


def finite_pearson(x: np.ndarray, y: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    if int(mask.sum()) < 5:
        return np.nan
    if np.nanstd(x[mask]) <= 0.0 or np.nanstd(y[mask]) <= 0.0:
        return np.nan
    return float(np.corrcoef(x[mask], y[mask])[0, 1])


def finite_spearman(x: np.ndarray, y: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    if int(mask.sum()) < 5:
        return np.nan
    rank_x = pd.Series(x[mask]).rank(method="average").to_numpy(dtype=float)
    rank_y = pd.Series(y[mask]).rank(method="average").to_numpy(dtype=float)
    return finite_pearson(rank_x, rank_y)


def set_log_column(df: pd.DataFrame, source_column: str, log_column: str) -> None:
    values = df[source_column].to_numpy(dtype=float)
    df[log_column] = np.nan
    positive = values > 0.0
    df.loc[positive, log_column] = np.log(values[positive])


def split_valid_indices(valid_indices: np.ndarray, min_valid_samples: int, max_segments: int) -> list[np.ndarray]:
    valid_indices = np.asarray(valid_indices, dtype=np.int64)
    if valid_indices.size < int(min_valid_samples):
        return []
    segment_count = int(min(max_segments, valid_indices.size // int(min_valid_samples)))
    segment_count = max(int(min_segments_per_well), segment_count)
    segments = [segment for segment in np.array_split(valid_indices, segment_count) if segment.size >= min_valid_samples]
    return segments


def segment_ls_gain(
    seismic_values: np.ndarray,
    synthetic_values: np.ndarray,
    *,
    eps: float,
    min_valid_samples: int,
) -> float:
    seismic_values = np.asarray(seismic_values, dtype=float)
    synthetic_values = np.asarray(synthetic_values, dtype=float)
    valid = np.isfinite(seismic_values) & np.isfinite(synthetic_values)
    if int(valid.sum()) < int(min_valid_samples):
        return np.nan
    numerator = float(np.sum(seismic_values[valid] * synthetic_values[valid]))
    denominator = float(np.sum(synthetic_values[valid] ** 2))
    gain = numerator / (denominator + float(eps) * max(int(valid.sum()), 1))
    return float(gain) if np.isfinite(gain) and gain > 0.0 else np.nan


def segment_attribute_values(seismic_values: np.ndarray) -> dict:
    seismic_values = np.asarray(seismic_values, dtype=float)
    finite = np.isfinite(seismic_values)
    if not np.any(finite):
        return {"seismic_rms": np.nan, "seismic_abs_mean": np.nan, "seismic_abs_p90": np.nan}
    values = seismic_values[finite]
    abs_values = np.abs(values)
    return {
        "seismic_rms": float(np.sqrt(np.nanmean(values**2))),
        "seismic_abs_mean": float(np.nanmean(abs_values)),
        "seismic_abs_p90": float(np.nanpercentile(abs_values, 90.0)),
    }


## 3) 构建自适应 segment 样本

In [ ]:
segment_rows = []

for _, row in ok_gain_df.iterrows():
    well_name = str(row["well_name"])
    gain_curve_path = resolve_artifact_path(row["gain_curve_path"])
    if not gain_curve_path.exists():
        raise FileNotFoundError(gain_curve_path)

    curve_df = pd.read_csv(gain_curve_path)
    required_columns = {
        "twt_s",
        "tvdss_m",
        "seismic_norm",
        "synthetic_raw",
        "gain_smooth",
        "eval_mask",
    }
    missing = required_columns - set(curve_df.columns)
    if missing:
        raise ValueError(f"Gain curve {gain_curve_path} missing columns: {sorted(missing)}")

    twt_s = curve_df["twt_s"].to_numpy(dtype=float)
    tvdss_m = curve_df["tvdss_m"].to_numpy(dtype=float)
    seismic_norm = curve_df["seismic_norm"].to_numpy(dtype=float)
    synthetic_raw = curve_df["synthetic_raw"].to_numpy(dtype=float)
    gain_smooth = curve_df["gain_smooth"].to_numpy(dtype=float)
    eval_mask = curve_df["eval_mask"].astype(bool).to_numpy()

    finite = (
        eval_mask
        & np.isfinite(twt_s)
        & np.isfinite(tvdss_m)
        & np.isfinite(seismic_norm)
        & np.isfinite(synthetic_raw)
    )
    valid_indices = np.flatnonzero(finite)
    segments = split_valid_indices(valid_indices, min_segment_valid_samples, max_segment_count)

    scale = float(row["scale"])
    eps = float(row["eps"]) if "eps" in row.index and np.isfinite(float(row["eps"])) else 0.0

    for segment_index, segment_idx in enumerate(segments):
        seg_seismic = seismic_norm[segment_idx]
        seg_synthetic = synthetic_raw[segment_idx]
        seg_gain_smooth = gain_smooth[segment_idx]
        finite_gain = np.isfinite(seg_gain_smooth) & (seg_gain_smooth > 0.0)

        gain_segment_ls = segment_ls_gain(
            seg_seismic,
            seg_synthetic,
            eps=eps,
            min_valid_samples=min_segment_valid_samples,
        )
        gain_segment_median = float(np.nanmedian(seg_gain_smooth[finite_gain])) if np.any(finite_gain) else np.nan
        relative_gain_segment_ls = gain_segment_ls / scale if np.isfinite(gain_segment_ls) and scale > 0.0 else np.nan
        relative_gain_segment_median = (
            gain_segment_median / scale if np.isfinite(gain_segment_median) and scale > 0.0 else np.nan
        )

        attribute_values = segment_attribute_values(seg_seismic)
        segment_rows.append(
            {
                "well_name": well_name,
                "source_corr": float(row.get("source_corr", np.nan)),
                "scale": scale,
                "segment_index": int(segment_index),
                "segment_count": int(len(segments)),
                "n_valid_samples": int(segment_idx.size),
                "twt_min_s": float(np.nanmin(twt_s[segment_idx])),
                "twt_max_s": float(np.nanmax(twt_s[segment_idx])),
                "tvdss_min_m": float(np.nanmin(tvdss_m[segment_idx])),
                "tvdss_max_m": float(np.nanmax(tvdss_m[segment_idx])),
                "gain_segment_ls": gain_segment_ls,
                "gain_segment_median": gain_segment_median,
                "relative_gain_segment_ls": relative_gain_segment_ls,
                "relative_gain_segment_median": relative_gain_segment_median,
                **attribute_values,
            }
        )

segment_samples_df = pd.DataFrame(segment_rows)
if segment_samples_df.empty:
    raise ValueError("No adaptive segment samples were built.")

for column in [
    "gain_segment_ls",
    "gain_segment_median",
    "relative_gain_segment_ls",
    "relative_gain_segment_median",
    "seismic_rms",
    "seismic_abs_mean",
    "seismic_abs_p90",
]:
    set_log_column(segment_samples_df, column, f"log_{column}")

segment_samples_df.to_csv(segment_samples_file, index=False)
print("Saved", segment_samples_file)
display(segment_samples_df.groupby("well_name").agg(n_segments=("segment_index", "count"), n_valid_samples=("n_valid_samples", "sum"), source_corr=("source_corr", "first")))
display(segment_samples_df.head())


## 4) 绝对 vs 相对目标的相关性比较

In [ ]:
attribute_columns = ["seismic_rms", "seismic_abs_mean", "seismic_abs_p90"]
target_columns = [
    "gain_segment_ls",
    "relative_gain_segment_ls",
    "gain_segment_median",
    "relative_gain_segment_median",
]

target_labels = {
    "gain_segment_ls": "absolute_ls",
    "relative_gain_segment_ls": "relative_ls",
    "gain_segment_median": "absolute_median",
    "relative_gain_segment_median": "relative_median",
}

metric_rows = []
for target_column in target_columns:
    y_column = f"log_{target_column}"
    for attribute_column in attribute_columns:
        x_column = f"log_{attribute_column}"
        x = segment_samples_df[x_column].to_numpy(dtype=float)
        y = segment_samples_df[y_column].to_numpy(dtype=float)
        mask = np.isfinite(x) & np.isfinite(y)
        metric_rows.append(
            {
                "scope": "all_wells",
                "well_name": "__all__",
                "target": target_labels[target_column],
                "target_column": target_column,
                "attribute": attribute_column,
                "n_samples": int(mask.sum()),
                "pearson": finite_pearson(x, y),
                "spearman": finite_spearman(x, y),
                "abs_pearson": abs(finite_pearson(x, y)) if np.isfinite(finite_pearson(x, y)) else np.nan,
                "abs_spearman": abs(finite_spearman(x, y)) if np.isfinite(finite_spearman(x, y)) else np.nan,
            }
        )
        for well_name, well_df in segment_samples_df.groupby("well_name"):
            x_well = well_df[x_column].to_numpy(dtype=float)
            y_well = well_df[y_column].to_numpy(dtype=float)
            mask_well = np.isfinite(x_well) & np.isfinite(y_well)
            metric_rows.append(
                {
                    "scope": "per_well",
                    "well_name": well_name,
                    "target": target_labels[target_column],
                    "target_column": target_column,
                    "attribute": attribute_column,
                    "n_samples": int(mask_well.sum()),
                    "pearson": finite_pearson(x_well, y_well),
                    "spearman": finite_spearman(x_well, y_well),
                    "abs_pearson": abs(finite_pearson(x_well, y_well)) if np.isfinite(finite_pearson(x_well, y_well)) else np.nan,
                    "abs_spearman": abs(finite_spearman(x_well, y_well)) if np.isfinite(finite_spearman(x_well, y_well)) else np.nan,
                }
            )

relationship_metrics_df = pd.DataFrame(metric_rows)
relationship_metrics_df.to_csv(relationship_metrics_file, index=False)

all_relationships_df = relationship_metrics_df.loc[relationship_metrics_df["scope"] == "all_wells"].copy()
best_relationships_df = all_relationships_df.sort_values(
    ["abs_spearman", "abs_pearson", "n_samples"], ascending=[False, False, False]
).reset_index(drop=True)
best_relationships_df.to_csv(best_relationships_file, index=False)

print("Saved", relationship_metrics_file)
print("Saved", best_relationships_file)
display(best_relationships_df)


## 5) 图形 QC

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), constrained_layout=True)
plot_df = all_relationships_df.copy()
plot_df["relationship"] = plot_df["target"] + "\n" + plot_df["attribute"].str.replace("seismic_", "", regex=False)

axes[0].bar(np.arange(len(plot_df)), plot_df["pearson"], color="tab:blue", alpha=0.75)
axes[0].axhline(0.0, color="black", lw=0.8)
axes[0].set_xticks(np.arange(len(plot_df)))
axes[0].set_xticklabels(plot_df["relationship"], rotation=65, ha="right", fontsize=8)
axes[0].set_ylabel("Pearson r")
axes[0].set_title("All wells: log(attribute) vs log(target)")
axes[0].grid(True, axis="y", alpha=0.25)

axes[1].bar(np.arange(len(plot_df)), plot_df["spearman"], color="tab:green", alpha=0.75)
axes[1].axhline(0.0, color="black", lw=0.8)
axes[1].set_xticks(np.arange(len(plot_df)))
axes[1].set_xticklabels(plot_df["relationship"], rotation=65, ha="right", fontsize=8)
axes[1].set_ylabel("Spearman rho")
axes[1].set_title("Rank relationship")
axes[1].grid(True, axis="y", alpha=0.25)

summary_fig = figure_dir / "qc_01_abs_vs_relative_relationship_bars.png"
fig.savefig(summary_fig, dpi=180, bbox_inches="tight")
plt.show()
print("Saved", summary_fig)

best_row = best_relationships_df.iloc[0]
best_target_column = str(best_row["target_column"])
best_attribute = str(best_row["attribute"])
fig, ax = plt.subplots(figsize=(6.8, 5.2), constrained_layout=True)
well_names = segment_samples_df["well_name"].drop_duplicates().tolist()
color_map = {well_name: plt.cm.tab10(idx % 10) for idx, well_name in enumerate(well_names)}  # type: ignore
for well_name, well_df in segment_samples_df.groupby("well_name"):
    ax.scatter(
        well_df[f"log_{best_attribute}"],
        well_df[f"log_{best_target_column}"],
        s=32,
        alpha=0.75,
        color=color_map[well_name],
        label=well_name,
    )
ax.set_xlabel(f"log({best_attribute})")
ax.set_ylabel(f"log({best_target_column})")
ax.set_title(f"Best all-well relationship: {best_row['target']} / {best_attribute}")
ax.legend(loc="best", fontsize=7)
ax.grid(True, alpha=0.25)
best_fig = figure_dir / "qc_02_best_abs_vs_relative_relationship_scatter.png"
fig.savefig(best_fig, dpi=180, bbox_inches="tight")
plt.show()
print("Saved", best_fig)


## 6) 输出清单

In [ ]:
print("Segment samples:")
print(segment_samples_file)
print("\nMetrics:")
print(relationship_metrics_file)
print(best_relationships_file)
print("\nFigures:")
for path in sorted(figure_dir.glob("*.png")):
    print(path)
